In [ ]:
# ----------------------------------------
# Project 05 - Microsoft Fabric Analytics Platform
# Notebook 01 - Bronze Source Profiling
# ----------------------------------------

source_path = "Files/bronze/meter_readings/*"

bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(source_path)
)

print("Bronze smart meter data loaded successfully.")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 3, Finished, Available, Finished, False)

Bronze smart meter data loaded successfully.


In [2]:
print("BRONZE SCHEMA")
print("-" * 50)

bronze_df.printSchema()

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 4, Finished, Available, Finished, False)

BRONZE SCHEMA
--------------------------------------------------
root
 |-- LCLid: string (nullable = true)
 |-- stdorToU: string (nullable = true)
 |-- DateTime: string (nullable = true)
 |-- KWH/hh (per half hour) : string (nullable = true)



In [3]:
display(bronze_df.limit(10))

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4f053296-a82d-4279-8f15-6c6cdf9e3175)

In [4]:
row_count = bronze_df.count()
column_count = len(bronze_df.columns)

print("BRONZE SOURCE SUMMARY")
print("-" * 50)
print(f"Rows: {row_count:,}")
print(f"Columns: {column_count}")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 6, Finished, Available, Finished, False)

BRONZE SOURCE SUMMARY
--------------------------------------------------
Rows: 1,000,000
Columns: 4


In [5]:
print("SOURCE COLUMNS")
print("-" * 50)

for column in bronze_df.columns:
    print(column)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 7, Finished, Available, Finished, False)

SOURCE COLUMNS
--------------------------------------------------
LCLid
stdorToU
DateTime
KWH/hh (per half hour) 


In [6]:
print("DISTINCT VALUES")
print("-" * 50)

household_count = bronze_df.select("LCLid").distinct().count()
tariff_count = bronze_df.select("stdorToU").distinct().count()

print(f"Distinct households: {household_count:,}")
print(f"Distinct tariff groups: {tariff_count:,}")

print("\nTariff values:")
bronze_df.select("stdorToU").distinct().show(truncate=False)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 8, Finished, Available, Finished, False)

DISTINCT VALUES
--------------------------------------------------
Distinct households: 30
Distinct tariff groups: 1

Tariff values:
+--------+
|stdorToU|
+--------+
|Std     |
+--------+



In [7]:
from pyspark.sql import functions as F

print("NULL COUNTS")
print("-" * 50)

null_counts = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
])

display(null_counts)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 9, Finished, Available, Finished, False)

NULL COUNTS
--------------------------------------------------


SynapseWidget(Synapse.DataFrame, 6056e328-22b5-4179-9df2-d3275df87714)

In [8]:
print("DUPLICATE ROW CHECK")
print("-" * 50)

distinct_row_count = bronze_df.distinct().count()
duplicate_row_count = row_count - distinct_row_count

print(f"Total rows: {row_count:,}")
print(f"Distinct rows: {distinct_row_count:,}")
print(f"Duplicate rows: {duplicate_row_count:,}")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 10, Finished, Available, Finished, False)

DUPLICATE ROW CHECK
--------------------------------------------------
Total rows: 1,000,000
Distinct rows: 999,312
Duplicate rows: 688


In [9]:
print("HOUSEHOLD + TIMESTAMP DUPLICATES")
print("-" * 50)

duplicate_keys = (
    bronze_df
    .groupBy("LCLid", "DateTime")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys.count()

print(f"Duplicate household/timestamp combinations: {duplicate_key_count:,}")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 11, Finished, Available, Finished, False)

HOUSEHOLD + TIMESTAMP DUPLICATES
--------------------------------------------------
Duplicate household/timestamp combinations: 688


In [10]:
print("TIMESTAMP PROFILE")
print("-" * 50)

timestamp_profile = (
    bronze_df
    .withColumn(
        "ParsedDateTime",
        F.to_timestamp("DateTime")
    )
)

timestamp_profile.select(
    F.min("ParsedDateTime").alias("MinDateTime"),
    F.max("ParsedDateTime").alias("MaxDateTime"),
    F.sum(F.col("ParsedDateTime").isNull().cast("int")).alias("InvalidTimestampRows")
).show(truncate=False)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 12, Finished, Available, Finished, False)

TIMESTAMP PROFILE
--------------------------------------------------
+-------------------+-------------------+--------------------+
|MinDateTime        |MaxDateTime        |InvalidTimestampRows|
+-------------------+-------------------+--------------------+
|2011-12-06 13:00:00|2014-02-28 00:00:00|0                   |
+-------------------+-------------------+--------------------+



In [11]:
consumption_col = "KWH/hh (per half hour) "

print("CONSUMPTION PROFILE")
print("-" * 50)

consumption_profile = (
    bronze_df
    .withColumn(
        "ConsumptionKWhNumeric",
        F.col(consumption_col).cast("double")
    )
)

consumption_profile.select(
    F.min("ConsumptionKWhNumeric").alias("MinConsumptionKWh"),
    F.max("ConsumptionKWhNumeric").alias("MaxConsumptionKWh"),
    F.avg("ConsumptionKWhNumeric").alias("AvgConsumptionKWh"),
    F.sum(F.col("ConsumptionKWhNumeric").isNull().cast("int")).alias("InvalidNumericRows")
).show(truncate=False)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 13, Finished, Available, Finished, False)

CONSUMPTION PROFILE
--------------------------------------------------
+-----------------+-----------------+-------------------+------------------+
|MinConsumptionKWh|MaxConsumptionKWh|AvgConsumptionKWh  |InvalidNumericRows|
+-----------------+-----------------+-------------------+------------------+
|0.0              |6.5279999        |0.23957973280016198|29                |
+-----------------+-----------------+-------------------+------------------+



In [12]:
negative_consumption_count = (
    consumption_profile
    .filter(F.col("ConsumptionKWhNumeric") < 0)
    .count()
)

print(f"Negative consumption rows: {negative_consumption_count:,}")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 14, Finished, Available, Finished, False)

Negative consumption rows: 0


In [13]:
invalid_consumption = (
    bronze_df
    .withColumn(
        "ConsumptionKWhNumeric",
        F.col(consumption_col).cast("double")
    )
    .filter(
        F.col("ConsumptionKWhNumeric").isNull()
    )
)

print("INVALID CONSUMPTION RECORDS")
print("-" * 50)
print(f"Rows: {invalid_consumption.count():,}")

display(invalid_consumption)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 15, Finished, Available, Finished, False)

INVALID CONSUMPTION RECORDS
--------------------------------------------------
Rows: 29


SynapseWidget(Synapse.DataFrame, 7c1dc7ef-a0cc-4706-8030-5b125f380523)

In [14]:
print("DISTINCT INVALID CONSUMPTION VALUES")
print("-" * 50)

invalid_consumption.groupBy(consumption_col).count().show(
    truncate=False
)

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 16, Finished, Available, Finished, False)

DISTINCT INVALID CONSUMPTION VALUES
--------------------------------------------------
+-----------------------+-----+
|KWH/hh (per half hour) |count|
+-----------------------+-----+
|Null                   |29   |
+-----------------------+-----+



In [15]:
print("DUPLICATE HOUSEHOLD/TIMESTAMP RECORDS")
print("-" * 50)

duplicate_records = (
    bronze_df
    .join(
        duplicate_keys.select("LCLid", "DateTime"),
        on=["LCLid", "DateTime"],
        how="inner"
    )
    .orderBy("LCLid", "DateTime")
)

display(duplicate_records.limit(20))

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 17, Finished, Available, Finished, False)

DUPLICATE HOUSEHOLD/TIMESTAMP RECORDS
--------------------------------------------------


SynapseWidget(Synapse.DataFrame, a0bb9b27-b418-4e2e-ba44-45bdcff6ea58)

In [16]:
duplicate_conflicts = (
    bronze_df
    .groupBy("LCLid", "DateTime")
    .agg(
        F.count("*").alias("RowCount"),
        F.countDistinct("stdorToU").alias("DistinctTariffCount"),
        F.countDistinct(consumption_col).alias("DistinctConsumptionCount")
    )
    .filter(
        (F.col("RowCount") > 1) &
        (
            (F.col("DistinctTariffCount") > 1) |
            (F.col("DistinctConsumptionCount") > 1)
        )
    )
)

conflict_count = duplicate_conflicts.count()

print("DUPLICATE KEY CONFLICT CHECK")
print("-" * 50)
print(f"Conflicting household/timestamp combinations: {conflict_count:,}")

StatementMeta(, 956a26b8-45a5-4a91-abfc-5ca5a3af9675, 18, Finished, Available, Finished, False)

DUPLICATE KEY CONFLICT CHECK
--------------------------------------------------
Conflicting household/timestamp combinations: 0
